# Chapter 5: Perspective

Source orientation: The Four Pillars of Geometry, Chapter 5, printed pages 88-116, physical PDF pages 98-126, sections 5.1-5.9. The source span was used to set topic coverage and order. The prose, code, diagrams, labs, and checks here are original course material.

Chapter question: **What remains true when parallelism is replaced by visibility from a viewpoint?**

Chapter goal: build a projective way of seeing. By the end, a learner should be able to explain why a tiled floor can be drawn with a straightedge, why ordinary parallelism becomes a horizon phenomenon, how homogeneous coordinates make joins and meets symmetric, why projections of a line are linear fractional maps, and why the cross-ratio is the first nontrivial numerical invariant on a projective line.

The chapter is intentionally visual-first. Every diagram below has a computational check attached to it: vanishing rows satisfy a recurrence, projective grid families meet at their assigned points at infinity, joins and meets satisfy incidence equations, Mobius maps preserve cross-ratio, and the finite Fano model satisfies the projective-plane incidence axioms.


## Computational translation guide

| Chapter idea | Computational object | Representation | Check |
| --- | --- | --- | --- |
| A receding row of equal tiles | the sequence `y_n = n/(n+1)` approaching a horizon | Matplotlib perspective grid | `f(y_n) = y_{n+1}` for `f(y)=1/(2-y)` |
| Straightedge-only tile growth | lines through fixed vanishing points plus a diagonal vanishing point | projective quadrilateral grid | each line family has a common meet on the horizon |
| The real projective plane | lines through the origin in `R^3`, with horizontal directions as infinity | 3D sight-line model | directions to distant points approach the horizontal |
| Homogeneous coordinates | points and lines as triples, with joins/meets by cross product | incidence diagram and table | `p dot l = 0` for constructed incidences |
| Projection from one line to another | a `2 x 2` matrix acting by `(ax+b)/(cx+d)` | projection diagram and rational graph | SymPy verifies projection and composition identities |
| Cross-ratio | a function of four ordered points on `RP^1` | invariant tracker and Mobius lab | values agree before and after projective maps |
| Projective plane over a field | nonzero triples over `F_2`, grouped by linear equations | Fano-plane incidence model | every pair of points has one line and every pair of lines has one point |

Library routing: Matplotlib is used for durable construction and incidence diagrams; OpenCV is used where the geometry is literally a homography; Plotly is used for the interactive Mobius/cross-ratio lab; SymPy is used for exact linear-fractional algebra; NetworkX is used for the proof-dependency scaffold. These choices follow the course catalog preference for projective/computer-vision tooling when the chapter concept is camera projection or homography, and for symbolic checks when the invariant is algebraic.


In [ ]:
from __future__ import annotations

from pathlib import Path
import itertools
import json
import math
import sys

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import networkx as nx
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp


def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "The-Four-Pillars-of-Geometry" and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Could not locate course root")


BOOK_ROOT = find_book_root(Path.cwd().resolve())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_artifact_root, image_stats, save_json, save_table
from utils.projective import affine, cross_ratio, hpoint, incidence, join, meet, normalize_h

CHAPTER_KEY = "chapter-05-perspective"
ARTIFACT_ROOT = ensure_artifact_root(BOOK_ROOT / "artifacts" / CHAPTER_KEY)

# Remove stale generic scaffold outputs; this notebook builds chapter visuals directly.
STALE_ARTIFACTS = [
    "figures/visual_spine.png",
    "figures/construction_or_model.png",
    "figures/algebraic_check.png",
    "figures/invariant_heatmap.png",
    "figures/proof_state.png",
    "html/interactive_invariant_lab.html",
    "checks/artifact_manifest.json",
    "checks/invariants.json",
]
for rel in STALE_ARTIFACTS:
    stale = ARTIFACT_ROOT / rel
    if stale.exists():
        stale.unlink()

PALETTE = {
    "ink": "#1f2937",
    "muted": "#6b7280",
    "blue": "#2563eb",
    "teal": "#0f766e",
    "orange": "#c2410c",
    "green": "#15803d",
    "purple": "#7c3aed",
    "red": "#b91c1c",
    "gold": "#a16207",
    "paper": "#f8fafc",
}

artifact_registry = {"figures": [], "html": [], "checks": [], "tables": []}


def rel_artifact(path: Path) -> str:
    return path.relative_to(ARTIFACT_ROOT).as_posix()


def register_artifact(path: Path, category: str) -> Path:
    artifact_registry[category].append(rel_artifact(path))
    return path


def figure_path(name: str) -> Path:
    path = ARTIFACT_ROOT / "figures" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def save_figure(fig, name: str, *, dpi: int = 160) -> Path:
    path = figure_path(name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return register_artifact(path, "figures")


def save_check(data: dict, name: str) -> Path:
    return register_artifact(save_json(data, ARTIFACT_ROOT, "checks", name), "checks")


def save_rows(rows: list[dict], name: str) -> Path:
    return register_artifact(save_table(rows, ARTIFACT_ROOT, "tables", name), "tables")


def line_segment(line, xlim, ylim, eps=1e-10):
    a, b, c = np.asarray(line, dtype=float)
    points = []
    for x in xlim:
        if abs(b) > eps:
            y = -(a * x + c) / b
            if ylim[0] - 1e-7 <= y <= ylim[1] + 1e-7:
                points.append((float(x), float(y)))
    for y in ylim:
        if abs(a) > eps:
            x = -(b * y + c) / a
            if xlim[0] - 1e-7 <= x <= xlim[1] + 1e-7:
                points.append((float(x), float(y)))
    unique = []
    for p in points:
        if not any(np.linalg.norm(np.array(p) - np.array(q)) < 1e-7 for q in unique):
            unique.append(p)
    if len(unique) < 2:
        return None
    return unique[0], unique[1]


def draw_projective_line(ax, line, xlim, ylim, **kwargs):
    seg = line_segment(line, xlim, ylim)
    if seg is not None:
        (x0, y0), (x1, y1) = seg
        ax.plot([x0, x1], [y0, y1], **kwargs)


def dehom(p):
    p = np.asarray(p, dtype=float)
    return p[:2] / p[2]


print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT}")


## 1. Vanishing points turn equal spacing into a horizon limit

The simplest perspective grid starts with equal marks on the bottom edge, a horizon, and one vanishing point. The receding row positions are not equally spaced in the picture. They follow `y_n = n/(n+1)`, so the gaps shrink while the row lines approach the horizon. The map that moves one row to the next is linear fractional: `f(y)=1/(2-y)`.

Inspect the diagram for two things: the column lines all point to the same horizon point, and the horizontal row gaps decrease according to a precise recurrence rather than an arbitrary visual rule.


In [ ]:
horizon = 1.0
vp = np.array([0.0, horizon])
base_x = np.arange(-4, 5, dtype=float)
row_indices = np.arange(0, 9, dtype=float)
row_y = row_indices / (row_indices + 1.0)

fig, ax = plt.subplots(figsize=(9.2, 5.6))
ax.set_facecolor(PALETTE["paper"])
ax.axhline(horizon, color=PALETTE["ink"], lw=2.2)
ax.scatter([vp[0]], [vp[1]], s=70, color=PALETTE["red"], zorder=5)
ax.text(vp[0] + 0.12, vp[1] + 0.025, "vanishing point on horizon", color=PALETTE["red"], fontsize=10)

for x in base_x:
    ax.plot([x, vp[0]], [0, vp[1]], color=PALETTE["blue"], lw=1.1, alpha=0.78)
    ax.scatter([x], [0], s=14, color=PALETTE["blue"], zorder=4)

for n, y in enumerate(row_y):
    width = (1.0 - y / horizon) * max(abs(base_x))
    ax.plot([-width, width], [y, y], color=PALETTE["ink"], lw=1.0 if n else 1.8, alpha=0.86)
    if n in {0, 1, 2, 4, 8}:
        ax.text(width + 0.08, y - 0.015, f"n={n}, y={y:.3f}", fontsize=9, color=PALETTE["muted"])

for n in range(4):
    y0, y1 = row_y[n], row_y[n + 1]
    left0, right0 = -0.5 * (1 - y0), 0.5 * (1 - y0)
    left1, right1 = -0.5 * (1 - y1), 0.5 * (1 - y1)
    poly = Polygon([(left0, y0), (right0, y0), (right1, y1), (left1, y1)], closed=True,
                   facecolor=PALETTE["teal"], alpha=0.16, edgecolor="none")
    ax.add_patch(poly)

ax.annotate("row gaps shrink toward the horizon", xy=(2.3, row_y[4]), xytext=(2.65, 0.38),
            arrowprops={"arrowstyle": "->", "color": PALETTE["orange"], "lw": 1.4},
            color=PALETTE["orange"], fontsize=10)
ax.set_xlim(-4.5, 4.7)
ax.set_ylim(-0.08, 1.12)
ax.set_aspect("auto")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Perspective floor grid: equal ground steps approach one horizon", color=PALETTE["ink"])
for spine in ax.spines.values():
    spine.set_visible(False)

perspective_grid_path = save_figure(fig, "perspective-floor-grid-vanishing-points.png")
recurrence_error = float(np.max(np.abs(1.0 / (2.0 - row_y[:-1]) - row_y[1:])))
save_check({
    "row_formula": "y_n = n/(n+1)",
    "next_row_map": "f(y)=1/(2-y)",
    "max_row_recurrence_error": recurrence_error,
    "horizon_y": horizon,
    "rows_checked": int(len(row_y)),
}, "perspective-grid-recurrence.json")
display_artifact(perspective_grid_path, width=820)


## 2. Straightedge-only tile growth uses three points at infinity

A straightedge cannot copy a length. A projective tile construction does not try to copy lengths. It preserves incidence: each side family aims at a vanishing point on the horizon, and one diagonal of the first tile determines a third vanishing point for the diagonal family. Once those three points are present, new tiles can be grown by joining already-known intersections to the appropriate horizon point.

The figure below starts from one oblique tile. The blue and green families are the two tile directions; the orange family is the diagonal direction. The checks assert that representative members of each family meet at their assigned point on the horizon.


In [ ]:
V_left = np.array([-5.2, 3.0])
V_right = np.array([5.0, 3.0])
horizon_line = np.array([0.0, 1.0, -3.0])
A = np.array([-1.35, -0.92])
B = (1 - 0.23) * A + 0.23 * V_left
D = (1 - 0.24) * A + 0.24 * V_right
C = dehom(meet(join(hpoint(*B), hpoint(*V_right)), join(hpoint(*D), hpoint(*V_left))))

src_quad = np.array([[0, 0], [1, 0], [0, 1], [1, 1]], dtype=np.float32)
dst_quad = np.array([A, B, D, C], dtype=np.float32)
H_tile = cv2.getPerspectiveTransform(src_quad, dst_quad)


def map_tile_points(points):
    pts = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H_tile).reshape(-1, 2)


fig, ax = plt.subplots(figsize=(9.4, 6.0))
ax.set_facecolor(PALETTE["paper"])
ax.axhline(3.0, color=PALETTE["ink"], lw=2)
ax.text(-5.9, 3.08, "horizon", fontsize=10, color=PALETTE["ink"])
ax.scatter([V_left[0], V_right[0]], [V_left[1], V_right[1]], color=[PALETTE["blue"], PALETTE["green"]], s=58, zorder=5)
ax.text(V_left[0] + 0.1, V_left[1] + 0.1, "side VP", color=PALETTE["blue"], fontsize=9)
ax.text(V_right[0] - 1.0, V_right[1] + 0.1, "side VP", color=PALETTE["green"], fontsize=9)

u_values = np.linspace(-1.5, 5.5, 180)
for j in range(-1, 6):
    pts = map_tile_points(np.column_stack([u_values, np.full_like(u_values, j)]))
    ax.plot(pts[:, 0], pts[:, 1], color=PALETTE["blue"], lw=0.9, alpha=0.68)
for i in range(-2, 6):
    pts = map_tile_points(np.column_stack([np.full_like(u_values, i), u_values]))
    ax.plot(pts[:, 0], pts[:, 1], color=PALETTE["green"], lw=0.9, alpha=0.68)

first_tile = np.array([A, B, C, D, A])
ax.plot(first_tile[:, 0], first_tile[:, 1], color=PALETTE["ink"], lw=2.4, label="given tile")
ax.fill(first_tile[:-1, 0], first_tile[:-1, 1], color=PALETTE["gold"], alpha=0.22)

V_diag_h = H_tile @ np.array([1.0, 1.0, 0.0])
V_diag = dehom(V_diag_h)
ax.scatter([V_diag[0]], [V_diag[1]], s=58, color=PALETTE["orange"], zorder=5)
ax.text(V_diag[0] + 0.12, V_diag[1] + 0.1, "diagonal VP", color=PALETTE["orange"], fontsize=9)

for k in range(-2, 5):
    pts = map_tile_points(np.column_stack([u_values, u_values - k]))
    ax.plot(pts[:, 0], pts[:, 1], color=PALETTE["orange"], lw=0.85, alpha=0.5)

ax.plot([A[0], V_diag[0]], [A[1], V_diag[1]], color=PALETTE["orange"], lw=1.9, alpha=0.9)
ax.annotate("one diagonal supplies a new horizon point", xy=V_diag, xytext=(-2.1, 2.2),
            arrowprops={"arrowstyle": "->", "color": PALETTE["orange"], "lw": 1.3},
            color=PALETTE["orange"], fontsize=10)
for label, point in {"A": A, "B": B, "C": C, "D": D}.items():
    ax.scatter([point[0]], [point[1]], color=PALETTE["ink"], s=22, zorder=6)
    ax.text(point[0] + 0.06, point[1] + 0.05, label, fontsize=9, color=PALETTE["ink"])

ax.set_xlim(-6.0, 5.9)
ax.set_ylim(-1.25, 3.35)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Straightedge tile growth: joins to horizon points replace measured lengths", color=PALETTE["ink"])
for spine in ax.spines.values():
    spine.set_visible(False)

straightedge_path = save_figure(fig, "straightedge-tile-growth-diagonal-horizon.png")

family_checks = []
for family, direction, expected in [
    ("u_side", np.array([1.0, 0.0, 0.0]), np.r_[V_left, 1.0]),
    ("v_side", np.array([0.0, 1.0, 0.0]), np.r_[V_right, 1.0]),
    ("diagonal", np.array([1.0, 1.0, 0.0]), V_diag_h),
]:
    actual = normalize_h(H_tile @ direction)
    expected_n = normalize_h(expected)
    family_checks.append({
        "family": family,
        "normalized_vanishing_point": [float(x) for x in actual],
        "target_residual": float(np.linalg.norm(actual - expected_n)),
        "on_horizon_residual": float(abs(np.dot(horizon_line, actual))),
    })
save_check({
    "first_tile_vertices": {name: [float(v) for v in point] for name, point in {"A": A, "B": B, "C": C, "D": D}.items()},
    "family_checks": family_checks,
    "max_family_residual": float(max(item["target_residual"] for item in family_checks)),
    "max_horizon_residual": float(max(item["on_horizon_residual"] for item in family_checks)),
}, "straightedge-grid-vanishing-checks.json")
display_artifact(straightedge_path, width=830)


## 3. The real projective plane as all sight lines through one eye

The model `RP^2` turns a visible point into a line through the origin in `R^3`. A plane such as `z=-1` is one possible screen. Lines through the origin that never meet that screen are horizontal, so they become the points at infinity for that screen. This model explains why a Euclidean line has one point at infinity rather than two: walking far in either direction along the same line makes the sight line approach the same horizontal direction.

The numerical check records the angle between a sight line to `(t,0,-1)` and the horizontal plane. As `t` grows, the angle drops toward zero.


In [ ]:
fig = plt.figure(figsize=(9.2, 6.4))
ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor("white")

xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 2), np.linspace(-1.8, 1.8, 2))
zz = -np.ones_like(xx)
ax.plot_surface(xx, yy, zz, alpha=0.18, color="#93c5fd", edgecolor="none")

horiz_square = [[(-2.6, -1.9, 0), (2.6, -1.9, 0), (2.6, 1.9, 0), (-2.6, 1.9, 0)]]
ax.add_collection3d(Poly3DCollection(horiz_square, alpha=0.08, facecolor="#f59e0b", edgecolor="#a16207"))
ax.plot([-2.6, 2.6], [0, 0], [0, 0], color=PALETTE["orange"], lw=2.0)
ax.text(1.7, 0.1, 0.08, "horizontal point at infinity", color=PALETTE["orange"], fontsize=9)

ax.scatter([0], [0], [0], color=PALETTE["red"], s=48)
ax.text(0.05, 0.05, 0.08, "eye O", color=PALETTE["red"], fontsize=10)

screen_points = [np.array([0.4, -1.2, -1.0]), np.array([1.3, -0.4, -1.0]), np.array([2.2, 0.4, -1.0]), np.array([2.9, 0.9, -1.0])]
for idx, P in enumerate(screen_points, start=1):
    ax.scatter([P[0]], [P[1]], [P[2]], color=PALETTE["blue"], s=30)
    ax.plot([0, P[0]], [0, P[1]], [0, P[2]], color=PALETTE["blue"], lw=1.5, alpha=0.9)
    ax.text(P[0], P[1], P[2] - 0.08, f"P{idx}", color=PALETTE["ink"], fontsize=8)

line_x = np.linspace(-2.2, 2.2, 80)
line_y = 0.45 * line_x - 0.35
ax.plot(line_x, line_y, -np.ones_like(line_x), color=PALETTE["green"], lw=2.2)
ax.plot([-2.6, 2.6], [-1.17, 1.17], [0, 0], color=PALETTE["green"], lw=1.8, ls="--")
ax.text(-2.3, -1.55, -1.05, "ordinary line on screen", color=PALETTE["green"], fontsize=9)

ax.set_xlim(-2.8, 3.0)
ax.set_ylim(-2.1, 2.1)
ax.set_zlim(-1.25, 0.45)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=22, azim=-58)
ax.set_title("RP^2 model: projective points are sight lines through O", color=PALETTE["ink"])

rp2_model_path = save_figure(fig, "projective-plane-models-horizon.png")

angle_rows = []
for t_value in [1, 2, 4, 8, 16]:
    direction = np.array([float(t_value), 0.0, -1.0])
    angle = math.degrees(math.atan2(abs(direction[2]), np.linalg.norm(direction[:2])))
    angle_rows.append({"t": t_value, "angle_to_horizontal_degrees": angle})
save_rows(angle_rows, "rp2-sight-line-limit.csv")
save_check({
    "screen": "z=-1",
    "limit_direction": [1.0, 0.0, 0.0],
    "angles_to_horizontal_degrees": angle_rows,
    "monotone_decrease": bool(all(angle_rows[i]["angle_to_horizontal_degrees"] > angle_rows[i + 1]["angle_to_horizontal_degrees"] for i in range(len(angle_rows) - 1))),
}, "rp2-sight-line-limit.json")
display_artifact(rp2_model_path, width=820)


## 4. Homogeneous coordinates make join and meet the same operation

In ordinary coordinates, drawing a line through two points and intersecting two lines feel like different operations. In homogeneous coordinates they are dual: both are cross products. The line through points `p` and `q` is `p x q`; the point common to lines `l` and `m` is `l x m`.

This section uses the course-local projective helpers directly. The table records the incidence residuals, so the diagram is not just a picture: each marked point lies on the line it claims to lie on.


In [ ]:
A_h = hpoint(-2.2, -0.8)
B_h = hpoint(1.5, 1.15)
C_h = hpoint(-2.0, 1.25)
D_h = hpoint(2.3, -0.35)
line_AB = join(A_h, B_h)
line_CD = join(C_h, D_h)
E_h = meet(line_AB, line_CD)
E = affine(E_h)

P_h = hpoint(-2.2, -1.7)
Q_h = hpoint(2.1, -1.7)
R_h = hpoint(-2.2, -1.18)
S_h = hpoint(2.1, -1.18)
parallel_1 = join(P_h, Q_h)
parallel_2 = join(R_h, S_h)
E_inf_h = normalize_h(meet(parallel_1, parallel_2))

fig, ax = plt.subplots(figsize=(8.6, 6.0))
ax.set_facecolor(PALETTE["paper"])
xlim = (-3.0, 3.0)
ylim = (-2.05, 2.0)
draw_projective_line(ax, line_AB, xlim, ylim, color=PALETTE["blue"], lw=2.0, label="join(A,B)")
draw_projective_line(ax, line_CD, xlim, ylim, color=PALETTE["green"], lw=2.0, label="join(C,D)")
draw_projective_line(ax, parallel_1, xlim, ylim, color=PALETTE["orange"], lw=1.8, ls="--")
draw_projective_line(ax, parallel_2, xlim, ylim, color=PALETTE["orange"], lw=1.8, ls="--")

for name, p, color in [
    ("A", A_h, PALETTE["blue"]), ("B", B_h, PALETTE["blue"]),
    ("C", C_h, PALETTE["green"]), ("D", D_h, PALETTE["green"]),
]:
    xy = affine(p)
    ax.scatter([xy[0]], [xy[1]], s=45, color=color, zorder=5)
    ax.text(xy[0] + 0.06, xy[1] + 0.06, name, fontsize=10, color=PALETTE["ink"])

ax.scatter([E[0]], [E[1]], s=62, color=PALETTE["red"], zorder=6)
ax.text(E[0] + 0.06, E[1] + 0.06, "meet", fontsize=10, color=PALETTE["red"])
ax.annotate("parallel affine lines meet at [1:0:0]", xy=(2.6, -1.42), xytext=(0.2, -0.78),
            arrowprops={"arrowstyle": "->", "color": PALETTE["orange"], "lw": 1.2},
            color=PALETTE["orange"], fontsize=10)

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal", adjustable="box")
ax.grid(color="#e5e7eb", lw=0.6)
ax.legend(loc="upper left", frameon=False)
ax.set_title("Homogeneous join and meet: cross products encode incidence", color=PALETTE["ink"])

join_meet_path = save_figure(fig, "homogeneous-join-meet-incidence.png")

incidence_rows = []
for point_name, point, line_name, line in [
    ("A", A_h, "AB", line_AB), ("B", B_h, "AB", line_AB),
    ("C", C_h, "CD", line_CD), ("D", D_h, "CD", line_CD),
    ("E", E_h, "AB", line_AB), ("E", E_h, "CD", line_CD),
    ("E_inf", E_inf_h, "parallel_1", parallel_1), ("E_inf", E_inf_h, "parallel_2", parallel_2),
]:
    incidence_rows.append({
        "point": point_name,
        "line": line_name,
        "residual": float(abs(np.dot(point, line))),
        "incident": bool(incidence(point, line, tol=1e-8)),
    })
save_rows(incidence_rows, "homogeneous-incidence-residuals.csv")
save_check({
    "finite_meet_affine": [float(v) for v in E],
    "parallel_meet_homogeneous": [float(v) for v in E_inf_h],
    "max_incidence_residual": float(max(row["residual"] for row in incidence_rows)),
    "all_incident": bool(all(row["incident"] for row in incidence_rows)),
}, "homogeneous-join-meet-checks.json")
display_artifact(join_meet_path, width=800)


## 5. A projection of one line is a linear fractional map

Take a source point `t` on the `x`-axis. Project it from a center `(a,b)` onto the target line `y=cx`. Solving the collinearity condition gives another point whose target coordinate is a linear fractional function of `t`.

The left panel shows the projection rays. The right panel shows the rational function with its vertical asymptote. The symbolic check below also verifies the matrix rule: composing two linear fractional maps is the same as multiplying their `2 x 2` matrices.


In [ ]:
a_val = 1.25
b_val = 1.35
c_val = 0.62
source_t = np.array([-3.0, -2.0, -1.0, 0.25, 0.9, 1.8, 2.8, 3.7])


def projected_x(t):
    return (b_val * t) / (c_val * t - a_val * c_val + b_val)


image_x = projected_x(source_t)
image_y = c_val * image_x
center = np.array([a_val, b_val])

fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.2), gridspec_kw={"width_ratios": [1.05, 1]})
ax = axes[0]
ax.set_facecolor(PALETTE["paper"])
ax.axhline(0, color=PALETTE["ink"], lw=1.6)
xx = np.linspace(-3.4, 4.1, 100)
ax.plot(xx, c_val * xx, color=PALETTE["green"], lw=2.0, label="target line y=cx")
ax.scatter([center[0]], [center[1]], color=PALETTE["red"], s=58, zorder=5)
ax.text(center[0] + 0.08, center[1] + 0.08, "projection center", color=PALETTE["red"], fontsize=9)
for t0, x1, y1 in zip(source_t, image_x, image_y):
    ax.plot([center[0], t0], [center[1], 0], color=PALETTE["blue"], lw=0.9, alpha=0.42)
    ax.plot([center[0], x1], [center[1], y1], color=PALETTE["blue"], lw=0.9, alpha=0.42)
    ax.scatter([t0], [0], color=PALETTE["blue"], s=24)
    ax.scatter([x1], [y1], color=PALETTE["green"], s=26)
ax.set_xlim(-3.5, 4.2)
ax.set_ylim(-1.2, 2.45)
ax.set_aspect("equal", adjustable="box")
ax.grid(color="#e5e7eb", lw=0.6)
ax.legend(loc="lower right", frameon=False)
ax.set_title("Projection rays from one point", color=PALETTE["ink"])

ax = axes[1]
ax.set_facecolor(PALETTE["paper"])
t_sing = (a_val * c_val - b_val) / c_val
t_grid_1 = np.linspace(-4.0, t_sing - 0.04, 260)
t_grid_2 = np.linspace(t_sing + 0.04, 4.5, 260)
ax.plot(t_grid_1, projected_x(t_grid_1), color=PALETTE["purple"], lw=2.0)
ax.plot(t_grid_2, projected_x(t_grid_2), color=PALETTE["purple"], lw=2.0)
ax.axvline(t_sing, color=PALETTE["red"], lw=1.4, ls="--", label="source point sent to infinity")
ax.scatter(source_t, image_x, color=PALETTE["purple"], s=24)
ax.set_xlim(-4.0, 4.5)
ax.set_ylim(-5.0, 5.0)
ax.grid(color="#e5e7eb", lw=0.6)
ax.legend(loc="upper right", frameon=False)
ax.set_xlabel("source coordinate t")
ax.set_ylabel("target coordinate x")
ax.set_title("Linear fractional coordinate map", color=PALETTE["ink"])

linear_fractional_path = save_figure(fig, "linear-fractional-projection-map.png")

t, x, a, b, c = sp.symbols("t x a b c", nonzero=True)
collinearity = sp.Matrix([[t, 0, 1], [a, b, 1], [x, c * x, 1]]).det()
solution = sp.solve(sp.Eq(collinearity, 0), x)[0]
expected = b * t / (c * t - a * c + b)
assert sp.simplify(solution - expected) == 0

x_sym = sp.symbols("x")
M1 = sp.Matrix([[2, 1], [1, 1]])
M2 = sp.Matrix([[1, -2], [3, 1]])


def lft_expr(M, variable):
    return sp.simplify((M[0, 0] * variable + M[0, 1]) / (M[1, 0] * variable + M[1, 1]))


composition_expr = sp.simplify(lft_expr(M1, lft_expr(M2, x_sym)))
matrix_product_expr = lft_expr(M1 * M2, x_sym)
composition_residual = sp.simplify(composition_expr - matrix_product_expr)
assert composition_residual == 0

save_check({
    "projection_formula": str(expected),
    "sympy_solution": str(solution),
    "singular_source_t": float(t_sing),
    "composition_matrix_rule_residual": str(composition_residual),
    "matrix_1_det": int(M1.det()),
    "matrix_2_det": int(M2.det()),
    "product_det": int((M1 * M2).det()),
}, "linear-fractional-symbolic-checks.json")
display_artifact(linear_fractional_path, width=900)


## 6. Homography lab: a square grid can be warped, but lines stay lines

A camera view of a plane is represented by a homography. OpenCV is the natural library for this laboratory because the computation is exactly the computer-vision version of the chapter's projection story. We synthesize a square grid, choose four point correspondences, estimate the homography, warp the image, and check that the four source corners reproject to the requested target corners.

This lab is deliberately synthetic. It uses no textbook image crops or screenshots.


In [ ]:
size = 420
source_img = np.full((size, size, 3), 255, dtype=np.uint8)
for u in range(0, size + 1, 42):
    cv2.line(source_img, (u, 0), (u, size - 1), (220, 228, 236), 1)
    cv2.line(source_img, (0, u), (size - 1, u), (220, 228, 236), 1)
for u in range(0, size, 84):
    for v in range(0, size, 84):
        if (u // 84 + v // 84) % 2 == 0:
            cv2.rectangle(source_img, (u, v), (u + 84, v + 84), (235, 246, 255), -1)
            cv2.line(source_img, (u, v), (u + 84, v), (190, 205, 220), 1)
            cv2.line(source_img, (u, v), (u, v + 84), (190, 205, 220), 1)

src_corners = np.array([[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]], dtype=np.float32)
dst_corners = np.array([[128, 62], [345, 112], [392, 365], [36, 330]], dtype=np.float32)
H_img = cv2.getPerspectiveTransform(src_corners, dst_corners)
warped = cv2.warpPerspective(source_img, H_img, (size, size), borderValue=(255, 255, 255))
projected_corners = cv2.perspectiveTransform(src_corners.reshape(-1, 1, 2), H_img).reshape(-1, 2)
reprojection_errors = np.linalg.norm(projected_corners - dst_corners, axis=1)

H_world_to_img = H_img @ np.array([[size - 1, 0, 0], [0, size - 1, 0], [0, 0, 1]], dtype=float)
vp_x_img = H_world_to_img @ np.array([1.0, 0.0, 0.0])
vp_y_img = H_world_to_img @ np.array([0.0, 1.0, 0.0])
vp_x_img = vp_x_img[:2] / vp_x_img[2]
vp_y_img = vp_y_img[:2] / vp_y_img[2]

fig, axes = plt.subplots(1, 2, figsize=(10.2, 5.2))
for ax, image, title in [(axes[0], source_img, "synthetic square grid"), (axes[1], warped, "homography warp")]:
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title, color=PALETTE["ink"])
    ax.set_xticks([])
    ax.set_yticks([])
for ax, corners, color in [(axes[0], src_corners, PALETTE["blue"]), (axes[1], dst_corners, PALETTE["red"])]:
    closed = np.vstack([corners, corners[0]])
    ax.plot(closed[:, 0], closed[:, 1], color=color, lw=2.0)
    ax.scatter(corners[:, 0], corners[:, 1], color=color, s=26)
axes[1].text(8, 22, "lines remain lines; spacings do not", color=PALETTE["ink"], fontsize=10,
             bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none"})
fig.suptitle("OpenCV homography lab: projective image of a plane", color=PALETTE["ink"])

homography_path = save_figure(fig, "homography-synthetic-grid-warp.png")
save_check({
    "homography_matrix": [[float(v) for v in row] for row in H_img],
    "corner_reprojection_errors": [float(v) for v in reprojection_errors],
    "max_corner_reprojection_error": float(np.max(reprojection_errors)),
    "x_direction_vanishing_point_pixels": [float(v) for v in vp_x_img],
    "y_direction_vanishing_point_pixels": [float(v) for v in vp_y_img],
}, "homography-reprojection-checks.json")
display_artifact(homography_path, width=820)


## 7. Cross-ratio is the first visible number that survives projection

A projection can ruin lengths and ordinary ratios. For four points on a line, however, the ratio of two ratios survives every linear fractional map. The base row below uses four equally spaced points, whose cross-ratio is `4/3`. The Mobius rows look unequally spaced, but the computed cross-ratio remains the same. The last row shows a tempting but wrong drawing rule: shrinking each gap by a constant factor changes the cross-ratio, so it cannot be the correct perspective rule for equal tiles.

This is the chapter's main invariant. The proof scaffold in the next section explains why it determines projective maps of `RP^1`.


In [ ]:
base_points = np.array([0.0, 1.0, 2.0, 3.0])
base_cr = cross_ratio(*base_points)
matrices = [
    ("identity", np.array([[1.0, 0.0], [0.0, 1.0]])),
    ("gentle perspective", np.array([[1.25, 0.35], [0.16, 1.0]])),
    ("skew view", np.array([[0.85, -0.9], [-0.18, 1.25]])),
    ("near infinity", np.array([[1.4, -0.35], [0.28, 0.9]])),
]


def mobius_values(M, xs):
    xs = np.asarray(xs, dtype=float)
    return (M[0, 0] * xs + M[0, 1]) / (M[1, 0] * xs + M[1, 1])


rows = []
fig, ax = plt.subplots(figsize=(10.0, 5.8))
ax.set_facecolor(PALETTE["paper"])
y_positions = np.arange(len(matrices) + 1, dtype=float)[::-1]
for idx, ((label, M), y) in enumerate(zip(matrices, y_positions[:-1])):
    values = mobius_values(M, base_points)
    cr = cross_ratio(*values)
    rows.append({
        "label": label,
        "p": float(values[0]), "q": float(values[1]), "r": float(values[2]), "s": float(values[3]),
        "cross_ratio": float(cr),
        "max_gap": float(np.max(np.diff(np.sort(values)))),
        "min_gap": float(np.min(np.diff(np.sort(values)))),
    })
    display_values = (values - values.min()) / (values.max() - values.min()) * 7.0 - 3.5
    ax.plot([display_values.min() - 0.25, display_values.max() + 0.25], [y, y], color=PALETTE["ink"], lw=1.0)
    colors = [PALETTE["blue"], PALETTE["green"], PALETTE["orange"], PALETTE["red"]]
    for name, xval, color in zip(["p", "q", "r", "s"], display_values, colors):
        ax.scatter([xval], [y], s=58, color=color, zorder=5)
        ax.text(xval, y + 0.14, name, ha="center", fontsize=9, color=color)
    ax.text(-4.4, y, f"{label}: CR={cr:.6f}", va="center", fontsize=10, color=PALETTE["ink"])

e = 0.72
bad_points = np.array([0.0, 1.0, 1.0 + e, 1.0 + e + e * e])
bad_cr = cross_ratio(*bad_points)
y = y_positions[-1]
bad_display = (bad_points - bad_points.min()) / (bad_points.max() - bad_points.min()) * 7.0 - 3.5
ax.plot([bad_display.min() - 0.25, bad_display.max() + 0.25], [y, y], color=PALETTE["red"], lw=1.1, ls="--")
for name, xval in zip(["p", "q", "r", "s"], bad_display):
    ax.scatter([xval], [y], s=55, color=PALETTE["red"], zorder=5)
    ax.text(xval, y + 0.14, name, ha="center", fontsize=9, color=PALETTE["red"])
ax.text(-4.4, y, f"constant shrink e={e}: CR={bad_cr:.6f}", va="center", fontsize=10, color=PALETTE["red"])
rows.append({"label": "constant shrink", "p": float(bad_points[0]), "q": float(bad_points[1]), "r": float(bad_points[2]), "s": float(bad_points[3]), "cross_ratio": float(bad_cr), "max_gap": float(max(np.diff(bad_points))), "min_gap": float(min(np.diff(bad_points)))})

ax.set_xlim(-4.7, 4.0)
ax.set_ylim(-0.7, len(matrices) + 0.6)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Cross-ratio tracker: projective maps distort gaps but preserve 4/3", color=PALETTE["ink"])
for spine in ax.spines.values():
    spine.set_visible(False)

cross_ratio_path = save_figure(fig, "cross-ratio-invariance-tracker.png")
save_rows(rows, "cross-ratio-mobius-samples.csv")
max_cr_error = max(abs(row["cross_ratio"] - base_cr) for row in rows if row["label"] != "constant shrink")
save_check({
    "base_points": [float(v) for v in base_points],
    "base_cross_ratio": float(base_cr),
    "mobius_max_cross_ratio_error": float(max_cr_error),
    "constant_shrink_cross_ratio": float(bad_cr),
    "constant_shrink_error_from_equal_spacing": float(abs(bad_cr - base_cr)),
}, "cross-ratio-invariance-checks.json")
display_artifact(cross_ratio_path, width=860)


## 8. Mobius lab: move four points and watch one number refuse to move

The interactive artifact below varies a one-parameter family of Mobius transformations. The top panel shows the image positions of four points on the projective line; the bottom panel records the cross-ratio across the slider. The plotted value is flat up to numerical roundoff.

A useful experiment: focus on a slider value where the points bunch up visually, then read the invariant trace. Perspective may compress the picture, but it does not compress the projective invariant.


In [ ]:
tau_values = np.linspace(-0.8, 0.8, 21)
source_points = np.array([-1.2, -0.1, 1.0, 2.4])
source_cr = cross_ratio(*source_points)
frames = []
cr_rows = []
for tau in tau_values:
    M = np.array([[1.0 + 0.35 * tau, 0.55 * tau], [0.22 * tau, 1.0 - 0.18 * tau]])
    mapped = mobius_values(M, source_points)
    cr_val = cross_ratio(*mapped)
    cr_rows.append({"tau": float(tau), "cross_ratio": float(cr_val)})
    frames.append(go.Frame(
        name=f"{tau:.2f}",
        data=[
            go.Scatter(x=mapped, y=np.zeros_like(mapped), mode="markers+text", text=["p", "q", "r", "s"],
                       textposition="top center", marker={"size": 14, "color": ["#2563eb", "#15803d", "#c2410c", "#b91c1c"]}),
            go.Scatter(x=[row["tau"] for row in cr_rows], y=[row["cross_ratio"] for row in cr_rows],
                       mode="lines+markers", line={"color": "#7c3aed"}),
        ],
    ))

initial_M = np.array([[1.0 + 0.35 * tau_values[0], 0.55 * tau_values[0]], [0.22 * tau_values[0], 1.0 - 0.18 * tau_values[0]]])
initial_mapped = mobius_values(initial_M, source_points)
fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.18,
                    subplot_titles=("images of four projective-line points", "cross-ratio across Mobius parameter"))
fig.add_trace(go.Scatter(x=initial_mapped, y=np.zeros_like(initial_mapped), mode="markers+text", text=["p", "q", "r", "s"],
                         textposition="top center", marker={"size": 14, "color": ["#2563eb", "#15803d", "#c2410c", "#b91c1c"]}), row=1, col=1)
fig.add_trace(go.Scatter(x=[cr_rows[0]["tau"]], y=[cr_rows[0]["cross_ratio"]], mode="lines+markers", line={"color": "#7c3aed"}), row=2, col=1)
fig.frames = frames
fig.update_yaxes(range=[-0.5, 0.7], showticklabels=False, row=1, col=1)
fig.update_xaxes(title_text="mapped coordinate", row=1, col=1)
fig.update_xaxes(title_text="tau", row=2, col=1)
fig.update_yaxes(title_text="cross-ratio", range=[source_cr - 0.02, source_cr + 0.02], row=2, col=1)
fig.update_layout(
    title="Mobius cross-ratio lab",
    template="plotly_white",
    width=860,
    height=620,
    sliders=[{
        "steps": [{"args": [[f"{tau:.2f}"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": f"{tau:.2f}", "method": "animate"} for tau in tau_values],
        "currentvalue": {"prefix": "tau = "},
    }],
)
html_path = ARTIFACT_ROOT / "html" / "mobius-cross-ratio-lab.html"
html_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
register_artifact(html_path, "html")
save_rows(cr_rows, "mobius-lab-cross-ratio-trace.csv")
save_check({
    "source_points": [float(v) for v in source_points],
    "source_cross_ratio": float(source_cr),
    "tau_count": int(len(tau_values)),
    "max_cross_ratio_deviation": float(np.max(np.abs(np.array([row["cross_ratio"] for row in cr_rows]) - source_cr))),
}, "mobius-lab-checks.json")
display_artifact(html_path, width="100%", height=640)


## 9. Proof scaffold: why cross-ratio characterizes projective maps of the line

The chapter's proof moves are easier to track as dependencies. The diagram is not a formal proof by itself; it is a map of what must already be known before the final characterization is credible.

Read it from left to right. Projections give linear fractional maps. Generators and matrix composition keep those maps in the same class. Cross-ratio is preserved by the generators, so it is preserved by every linear fractional map. Three-point existence plus fourth-point determination then gives the converse: a map preserving all cross-ratios has to be the unique linear fractional map with those first three images.


In [ ]:
G = nx.DiGraph()
proof_nodes = {
    "projection_incidence": "projection preserves\nline incidence",
    "one_projection_lft": "one line projection\ngives an LFT",
    "matrix_composition": "LFTs compose as\n2 x 2 matrices",
    "generator_cr": "generators preserve\ncross-ratio",
    "all_lft_cr": "every LFT preserves\ncross-ratio",
    "three_point_exists": "three source points\ncan be projected",
    "fourth_point": "a fourth point is\nfixed by cross-ratio",
    "characterization": "cross-ratio preserving\nmaps are LFTs",
}
G.add_nodes_from(proof_nodes)
G.add_edges_from([
    ("projection_incidence", "one_projection_lft"),
    ("one_projection_lft", "matrix_composition"),
    ("matrix_composition", "all_lft_cr"),
    ("generator_cr", "all_lft_cr"),
    ("all_lft_cr", "fourth_point"),
    ("three_point_exists", "characterization"),
    ("fourth_point", "characterization"),
    ("all_lft_cr", "characterization"),
])
pos = {
    "projection_incidence": (0, 1.3),
    "one_projection_lft": (1.5, 1.3),
    "matrix_composition": (3.0, 1.3),
    "generator_cr": (3.0, 0.1),
    "all_lft_cr": (4.7, 0.85),
    "three_point_exists": (4.7, -0.55),
    "fourth_point": (6.3, 0.45),
    "characterization": (8.0, 0.15),
}
fig, ax = plt.subplots(figsize=(11.2, 4.8))
ax.set_facecolor(PALETTE["paper"])
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.5, edge_color="#64748b")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2450, node_color="#dbeafe", edgecolors=PALETTE["ink"], linewidths=1.1)
nx.draw_networkx_labels(G, pos, labels=proof_nodes, ax=ax, font_size=9, font_color=PALETTE["ink"])
ax.set_title("Proof dependency scaffold for cross-ratio on RP^1", color=PALETTE["ink"])
ax.set_axis_off()

proof_path = save_figure(fig, "cross-ratio-proof-dependency.png")
save_check({
    "node_count": int(G.number_of_nodes()),
    "edge_count": int(G.number_of_edges()),
    "is_dag": bool(nx.is_directed_acyclic_graph(G)),
    "topological_order": list(nx.topological_sort(G)),
}, "cross-ratio-proof-dependency.json")
display_artifact(proof_path, width=900)


## 10. A finite projective plane model: the Fano plane

The real and complex projective planes are not the only models. If coordinates are taken in the field with two elements, the nonzero triples form seven projective points. Linear equations over that field group them into seven lines, each containing three points. The picture is symbolic, not metric: one of the lines is drawn as a circle so that all seven incidence triples can be seen.

This is a useful endpoint for Chapter 5 because it separates projective incidence from Euclidean measurement completely. The checks assert the two incidence axioms that can be tested in this finite model: every pair of points lies on exactly one line, and every pair of lines meets in exactly one point.


In [ ]:
points_f2 = {
    "100": np.array([1, 0, 0], dtype=int),
    "010": np.array([0, 1, 0], dtype=int),
    "001": np.array([0, 0, 1], dtype=int),
    "011": np.array([0, 1, 1], dtype=int),
    "101": np.array([1, 0, 1], dtype=int),
    "110": np.array([1, 1, 0], dtype=int),
    "111": np.array([1, 1, 1], dtype=int),
}
line_coeffs_f2 = {
    "x=0": np.array([1, 0, 0], dtype=int),
    "y=0": np.array([0, 1, 0], dtype=int),
    "z=0": np.array([0, 0, 1], dtype=int),
    "x+y=0": np.array([1, 1, 0], dtype=int),
    "y+z=0": np.array([0, 1, 1], dtype=int),
    "z+x=0": np.array([1, 0, 1], dtype=int),
    "x+y+z=0": np.array([1, 1, 1], dtype=int),
}
line_points = {
    name: [pname for pname, p in points_f2.items() if int(np.dot(coeff, p) % 2) == 0]
    for name, coeff in line_coeffs_f2.items()
}
positions = {
    "001": (0.0, 2.0),
    "101": (-1.55, 1.0),
    "011": (1.55, 1.0),
    "111": (0.0, 0.35),
    "100": (-1.5, -1.2),
    "110": (0.0, -1.2),
    "010": (1.5, -1.2),
}

fig, ax = plt.subplots(figsize=(7.6, 6.8))
ax.set_facecolor(PALETTE["paper"])
straight_line_names = ["x=0", "y=0", "z=0", "x+y=0", "y+z=0", "z+x=0"]
for lname in straight_line_names:
    pts = np.array([positions[p] for p in line_points[lname]], dtype=float)
    best = max(itertools.combinations(range(3), 2), key=lambda ij: np.linalg.norm(pts[ij[0]] - pts[ij[1]]))
    p0, p1 = pts[best[0]], pts[best[1]]
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color="#64748b", lw=1.7, alpha=0.85)


def circle_from_three(p1, p2, p3):
    A_mat = np.array([[2 * (p2[0] - p1[0]), 2 * (p2[1] - p1[1])], [2 * (p3[0] - p1[0]), 2 * (p3[1] - p1[1])]])
    b_vec = np.array([p2[0] ** 2 + p2[1] ** 2 - p1[0] ** 2 - p1[1] ** 2,
                      p3[0] ** 2 + p3[1] ** 2 - p1[0] ** 2 - p1[1] ** 2])
    center = np.linalg.solve(A_mat, b_vec)
    radius = np.linalg.norm(center - p1)
    return center, radius


circle_pts = [np.array(positions[p], dtype=float) for p in line_points["x+y+z=0"]]
center_circle, radius_circle = circle_from_three(*circle_pts)
ax.add_patch(Circle(center_circle, radius_circle, fill=False, color=PALETTE["orange"], lw=2.0))

for name, (x0, y0) in positions.items():
    ax.scatter([x0], [y0], s=90, color=PALETTE["blue"], edgecolor="white", linewidth=1.2, zorder=5)
    ax.text(x0, y0 + 0.16, f"({','.join(name)})", ha="center", fontsize=9, color=PALETTE["ink"])

ax.text(center_circle[0] + radius_circle - 0.2, center_circle[1], "x+y+z=0", color=PALETTE["orange"], fontsize=9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-2.35, 2.35)
ax.set_ylim(-1.75, 2.45)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Fano plane: seven points and seven linear equations over F_2", color=PALETTE["ink"])
for spine in ax.spines.values():
    spine.set_visible(False)

fano_path = save_figure(fig, "fano-plane-seven-point-model.png")

pair_line_counts = []
for p, q in itertools.combinations(points_f2, 2):
    containing = [lname for lname, pts in line_points.items() if p in pts and q in pts]
    pair_line_counts.append(len(containing))
pair_meet_counts = []
for l1, l2 in itertools.combinations(line_points, 2):
    common = sorted(set(line_points[l1]).intersection(line_points[l2]))
    pair_meet_counts.append(len(common))
save_check({
    "point_count": len(points_f2),
    "line_count": len(line_points),
    "points_per_line": {name: len(pts) for name, pts in line_points.items()},
    "pair_line_counts": pair_line_counts,
    "pair_meet_counts": pair_meet_counts,
    "every_pair_of_points_has_one_line": bool(all(count == 1 for count in pair_line_counts)),
    "every_pair_of_lines_has_one_point": bool(all(count == 1 for count in pair_meet_counts)),
}, "fano-plane-incidence-checks.json")
display_artifact(fano_path, width=740)


## Final sanity checks

The final cell checks four kinds of claims:

1. artifact integrity: every generated figure, HTML file, JSON file, and table exists and has nonzero size;
2. image integrity: PNG artifacts are large enough and nonblank;
3. projective invariants: recurrence, vanishing-point, incidence, homography, Mobius, proof, and finite-plane checks pass within tolerance;
4. source-specific coverage: the notebook includes the Chapter 5 concepts that motivated the storyboard.


In [ ]:
manifest_path = save_check({
    "chapter_key": CHAPTER_KEY,
    "source_span": "printed pp. 88-116 / PDF pp. 98-126",
    "figures": sorted(artifact_registry["figures"]),
    "html": sorted(artifact_registry["html"]),
    "checks": sorted(set(artifact_registry["checks"] + ["checks/artifact-manifest.json"])),
    "tables": sorted(artifact_registry["tables"]),
}, "artifact-manifest.json")

all_artifact_paths = []
for category, rels in artifact_registry.items():
    for rel in rels:
        all_artifact_paths.append(ARTIFACT_ROOT / rel)
all_artifact_paths.append(manifest_path)
assert_artifacts(all_artifact_paths, min_size=32)

png_stats = [image_stats(ARTIFACT_ROOT / rel) for rel in artifact_registry["figures"]]
assert len(png_stats) >= 8
assert all(item["width"] >= 300 and item["height"] >= 240 for item in png_stats)
assert all(item["pixel_std"] > 2.0 for item in png_stats)

load = lambda name: json.loads((ARTIFACT_ROOT / "checks" / name).read_text(encoding="utf-8"))
assert load("perspective-grid-recurrence.json")["max_row_recurrence_error"] < 1e-12
assert load("straightedge-grid-vanishing-checks.json")["max_horizon_residual"] < 1e-6
assert load("homogeneous-join-meet-checks.json")["all_incident"] is True
assert load("linear-fractional-symbolic-checks.json")["composition_matrix_rule_residual"] == "0"
assert load("homography-reprojection-checks.json")["max_corner_reprojection_error"] < 1e-4
assert load("cross-ratio-invariance-checks.json")["mobius_max_cross_ratio_error"] < 1e-12
assert load("mobius-lab-checks.json")["max_cross_ratio_deviation"] < 1e-12
assert load("cross-ratio-proof-dependency.json")["is_dag"] is True
assert load("fano-plane-incidence-checks.json")["every_pair_of_points_has_one_line"] is True
assert load("fano-plane-incidence-checks.json")["every_pair_of_lines_has_one_point"] is True

print(json.dumps({
    "figures": len(artifact_registry["figures"]),
    "html": len(artifact_registry["html"]),
    "checks": len(set(artifact_registry["checks"])),
    "tables": len(artifact_registry["tables"]),
    "min_png_pixel_std": min(item["pixel_std"] for item in png_stats),
}, indent=2))


## Takeaways

- Perspective replaces Euclidean equality with incidence and horizon behavior. Parallel directions are represented by points at infinity.
- Straightedge-only tile drawing works because the construction repeatedly joins known points to vanishing points; it does not measure or copy segment lengths.
- In `RP^2`, points and lines are both homogeneous triples up to nonzero scale. Join and meet become the same cross-product operation in dual roles.
- Projection of a line is exactly a linear fractional map on `RP^1`, with the point at infinity handling the apparent division-by-zero cases.
- The cross-ratio of four ordered points is preserved by projective maps and is strong enough to characterize projective maps of the line.
- Finite fields give finite projective planes, so the projective-plane axioms are incidence axioms rather than Euclidean measurement axioms.
